# Sentiment Analysis on Tweets

In this lab, we classify tweets into three categories: **negative**, **neutral** or **positive**.

We use the [TweetEval](https://huggingface.co/datasets/cardiffnlp/tweet_eval) dataset (*sentiment* task): about 60,000 English tweets, annotated by hand.

Outline:

1. tokenize tweets with spaCy and build a vocabulary;
2. prepare batches with a PyTorch `Dataset` and `DataLoader` (padding included);
3. train a bidirectional LSTM with a hand-written PyTorch training loop;
4. compare the result with a TF-IDF baseline and a pretrained transformer.

*Remember to enable the GPU: Runtime > Change runtime type > GPU.*

## Imports

In [ ]:
!pip install -q datasets

In [ ]:
import collections
import copy

import matplotlib.pyplot as plt
import numpy
import pandas
import spacy
import torch
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, recall_score
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Computing on: {device}")
torch.manual_seed(0)

## Loading the data

Hugging Face's `datasets` library downloads the dataset already split into training, validation and test sets.

In [ ]:
tweets = load_dataset("cardiffnlp/tweet_eval", "sentiment")
label_names = ["negative", "neutral", "positive"]
print(tweets)

*Explore the data:*

- *Print about ten tweets from the training set with their label. How were user names handled?*
- *Are the classes balanced? What does it imply for the choice of metric?*
- *What is the typical length of a tweet, in words?*

*Tip: `tweets["train"].to_pandas()` returns a pandas `DataFrame`.*

In [ ]:
# Your code here

### Solution

In [ ]:
train_df = tweets["train"].to_pandas()
train_df["sentiment"] = train_df.label.map(dict(enumerate(label_names)))
print(train_df.sample(10, random_state=0)[["sentiment", "text"]].to_string())

train_df.sentiment.value_counts().plot.bar(title="Class distribution")
plt.show()

train_df.text.str.split().str.len().plot.hist(bins=30, title="Tweet length (words)")
plt.show()

- User names are replaced with `@user`: the tweets are anonymized.
- The *neutral* class is the most frequent, the *negative* class the rarest: accuracy would favor a model that mostly predicts *neutral*. TweetEval's official metric is the **per-class average recall** (*macro recall*), which gives the same weight to each class.
- A tweet is about twenty words long, rarely more than 40.

## Baseline: TF-IDF and logistic regression

Before training a neural network, it is always worth measuring what a simple model achieves. If the network doesn't do better, it's useless!

*Train a logistic regression on a TF-IDF representation of the tweets (`TfidfVectorizer`, `LogisticRegression(max_iter=1000)`) and compute its macro recall on the validation set with `recall_score(..., average="macro")`.*

In [ ]:
# Your code here

### Solution

In [ ]:
vectorizer = TfidfVectorizer(min_df=2, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(tweets["train"]["text"])
X_val_tfidf = vectorizer.transform(tweets["validation"]["text"])

baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train_tfidf, tweets["train"]["label"])
baseline_recall = recall_score(tweets["validation"]["label"],
                               baseline.predict(X_val_tfidf), average="macro")
print(f"TF-IDF macro recall: {baseline_recall:.3f}")

## Tokenization with spaCy

For a recurrent network, each tweet must become a sequence of word indices. We use spaCy's tokenizer, which handles punctuation, emoticons and contractions much better than a simple `split()`.

`spacy.blank("en")` creates an empty English pipeline: it only contains the tokenizer, which is very fast.

In [ ]:
nlp = spacy.blank("en")


def tokenize(text: str) -> list[str]:
  return [token.lower_ for token in nlp.tokenizer(text) if not token.is_space]


print(tokenize(tweets["train"][0]["text"]))

## Building the vocabulary

*Build the vocabulary from the training set **only**:*

- *index `0` is reserved for padding (`<pad>`), index `1` for unknown words (`<unk>`);*
- *only keep words seen at least twice;*
- *write the function `numericalize(text) -> list[int]` that turns a tweet into a list of indices.*

*Why not use the validation and test sets to build the vocabulary?*

In [ ]:
# Your code here

### Solution

In [ ]:
counts = collections.Counter(
    word for text in tweets["train"]["text"] for word in tokenize(text))
vocab = {"<pad>": 0, "<unk>": 1}
for word, count in counts.most_common():
  if count >= 2:
    vocab[word] = len(vocab)
print(f"Vocabulary size: {len(vocab)}")


def numericalize(text: str) -> list[int]:
  return [vocab.get(word, vocab["<unk>"]) for word in tokenize(text)]


print(numericalize(tweets["train"][0]["text"]))

The vocabulary is part of the model: building it with validation or test data would "leak" information about these data, and the evaluation would be too optimistic.

## `Dataset` and `DataLoader`

*Write:*

- *a `TweetDataset(Dataset)` class whose `__getitem__` returns the pair `(tensor of indices, label)`;*
- *a `collate(batch)` function that pads the sequences (`pad_sequence`) and returns `(indices, lengths, labels)`;*
- *the three `DataLoader`s (shuffled training, unshuffled validation and test), with batches of 64.*

In [ ]:
# Your code here

### Solution

In [ ]:
class TweetDataset(Dataset):
  def __init__(self, split) -> None:
    self.sequences = [torch.tensor(numericalize(text) or [vocab["<unk>"]])
                      for text in split["text"]]
    self.labels = split["label"]

  def __len__(self) -> int:
    return len(self.labels)

  def __getitem__(self, i: int) -> tuple[torch.Tensor, int]:
    return self.sequences[i], self.labels[i]


def collate(batch):
  sequences, labels = zip(*batch)
  lengths = torch.tensor([len(sequence) for sequence in sequences])
  padded = pad_sequence(sequences, batch_first=True, padding_value=vocab["<pad>"])
  return padded, lengths, torch.tensor(labels)


train_loader = DataLoader(TweetDataset(tweets["train"]), batch_size=64,
                          shuffle=True, collate_fn=collate)
val_loader = DataLoader(TweetDataset(tweets["validation"]), batch_size=64,
                        collate_fn=collate)
test_loader = DataLoader(TweetDataset(tweets["test"]), batch_size=64,
                         collate_fn=collate)

ids, lengths, labels = next(iter(train_loader))
print(ids.shape, lengths[:8], labels[:8])

## The model: a bidirectional LSTM

*Complete the following model:*

- *an `nn.Embedding` layer (with `padding_idx`);*
- *a bidirectional `nn.LSTM`;*
- *dropout, then a linear layer that outputs one **logit** per class (no softmax: `cross_entropy` takes care of it).*

*In `forward`, use `pack_padded_sequence` so that the LSTM ignores padding, then concatenate the last hidden states of both directions (`h_n[-2]` and `h_n[-1]`).*

*Beware: `pack_padded_sequence` expects lengths on the CPU.*

In [ ]:
class SentimentLSTM(nn.Module):
  def __init__(self, vocab_size: int, embedding_dim: int = 128,
               hidden_size: int = 128, n_classes: int = 3) -> None:
    super().__init__()
    # Your code here

  def forward(self, ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    # Your code here
    pass

### Solution

In [ ]:
class SentimentLSTM(nn.Module):
  def __init__(self, vocab_size: int, embedding_dim: int = 128,
               hidden_size: int = 128, n_classes: int = 3) -> None:
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
    self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True,
                        bidirectional=True)
    self.dropout = nn.Dropout(0.3)
    self.head = nn.Linear(2 * hidden_size, n_classes)

  def forward(self, ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    embedded = self.dropout(self.embedding(ids))
    packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True,
                                  enforce_sorted=False)
    _, (h_n, _) = self.lstm(packed)
    # Last hidden states of the forward and backward directions
    summary = torch.cat([h_n[-2], h_n[-1]], dim=1)
    return self.head(self.dropout(summary))


model = SentimentLSTM(len(vocab)).to(device)
print(model)
print(sum(p.numel() for p in model.parameters()), "parameters")

## Evaluation loop

*Write a function `predict(model, loader) -> tuple[numpy.ndarray, numpy.ndarray]` that returns the predicted labels and the true labels of a whole `DataLoader`.*

*Don't forget `model.eval()`, `torch.no_grad()`, and moving tensors to `device`.*

In [ ]:
# Your code here

### Solution

In [ ]:
def predict(model: nn.Module, loader: DataLoader) -> tuple[numpy.ndarray, numpy.ndarray]:
  model.eval()
  predictions, truths = [], []
  with torch.no_grad():
    for ids, lengths, labels in loader:
      logits = model(ids.to(device), lengths)
      predictions.append(logits.argmax(dim=1).cpu())
      truths.append(labels)
  return torch.cat(predictions).numpy(), torch.cat(truths).numpy()


def macro_recall(model: nn.Module, loader: DataLoader) -> float:
  predictions, truths = predict(model, loader)
  return recall_score(truths, predictions, average="macro")


print(f"Macro recall before training: {macro_recall(model, val_loader):.3f}")

## Training loop

*Write the training loop:*

- *`AdamW` optimizer with a learning rate of `1e-3`;*
- *`nn.functional.cross_entropy` loss;*
- *at the end of each epoch, compute the macro recall on the validation set;*
- *keep a copy of the best model's weights (`copy.deepcopy(model.state_dict())`) and stop training if validation doesn't improve for 2 epochs (*early stopping*);*
- *at the end, reload the best weights with `load_state_dict`.*

*About ten epochs are enough.*

In [ ]:
# Your code here

### Solution

In [ ]:
model = SentimentLSTM(len(vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

best_recall, best_state, patience = 0.0, None, 0
for epoch in range(10):
  model.train()
  total_loss = 0.0
  for ids, lengths, labels in train_loader:
    ids, labels = ids.to(device), labels.to(device)
    optimizer.zero_grad()
    loss = nn.functional.cross_entropy(model(ids, lengths), labels)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()

  val_recall = macro_recall(model, val_loader)
  print(f"Epoch {epoch + 1}: loss {total_loss / len(train_loader):.3f}, "
        f"validation macro recall {val_recall:.3f}")
  if val_recall > best_recall:
    best_recall, best_state, patience = val_recall, copy.deepcopy(model.state_dict()), 0
  else:
    patience += 1
    if patience == 2:
      print("Early stopping")
      break

model.load_state_dict(best_state)
torch.save(model.state_dict(), "sentiment-lstm.pt")
print(f"Best validation macro recall: {best_recall:.3f}")

## Final evaluation on the test set

*Evaluate the best model on the test set: classification report (`classification_report`) and confusion matrix (`ConfusionMatrixDisplay.from_predictions`). Do the same for the TF-IDF baseline.*

*Which classes are most often confused? Does the LSTM beat the baseline?*

In [ ]:
# Your code here

### Solution

In [ ]:
predictions, truths = predict(model, test_loader)
print("Bidirectional LSTM")
print(classification_report(truths, predictions, target_names=label_names, digits=3))
ConfusionMatrixDisplay.from_predictions(truths, predictions, display_labels=label_names,
                                        normalize="true")
plt.show()

baseline_predictions = baseline.predict(vectorizer.transform(tweets["test"]["text"]))
print("TF-IDF baseline")
print(classification_report(truths, baseline_predictions, target_names=label_names, digits=3))

- Confusions mostly involve the *neutral* class; *negative* and *positive* are rarely confused with each other.
- The LSTM trained from scratch is only slightly better than the TF-IDF baseline (macro recall of about 0.59 against 0.56 on the test set): with only a few tens of thousands of tweets, it lacks data to learn the meaning of words. This is the motivation for pretrained models.

## For comparison: a pretrained transformer

The [`cardiffnlp/twitter-roberta-base-sentiment-latest`](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest) model is a RoBERTa pretrained on millions of tweets, then fine-tuned on this same task. The `transformers` `pipeline` function makes it usable in one line.

*Evaluate it on the first 1,000 test tweets and compare its macro recall with the LSTM's on the same tweets. What do you conclude?*

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis",
                      model="cardiffnlp/twitter-roberta-base-sentiment-latest",
                      device=0 if device == "cuda" else -1)
print(classifier("I love this course on PyTorch!"))

In [ ]:
# Your code here

### Solution

In [ ]:
subset = tweets["test"].select(range(1000))
name_to_label = {"negative": 0, "neutral": 1, "positive": 2}
roberta_predictions = [name_to_label[output["label"]]
                       for output in classifier(list(subset["text"]), batch_size=64)]
print(f"Pretrained RoBERTa: "
      f"{recall_score(list(subset['label']), roberta_predictions, average='macro'):.3f}")
print(f"LSTM: {recall_score(truths[:1000], predictions[:1000], average='macro'):.3f}")

The pretrained transformer does much better (about 0.71 against 0.59): it learned the language of tweets from millions of examples before being fine-tuned. For a common text classification task, starting from a pretrained model is today's first reflex; training an LSTM from scratch remains useful under tight compute constraints, or when no pretrained model exists for the language or domain.